# 06.5 — `monad3_c.bin`'s update mechanism, in C

**Fourth Age Paper companion notebook.** `ScalarContextPropagation` — §6
addendum. Separate from the Python-side code shown elsewhere in this
paper series (`monad.py::Crank.learn`) on purpose: the live Monad and
Monad Harness are C (`PtolC/monad.c`, what the daemon actually calls),
not Python — Python monads exist for testing and for CS-paper
readability, but they are a different, simpler formula, not a twin. This
notebook features **only the update mechanism itself** — the arithmetic
that runs when the Monad "learns" a word — not the C daemon that
triggers it (`PtolC/daemon.c`: sockets, FIFO listening, spool files).
That's normal everyday plumbing common to most server and desktop
software; it isn't this paper's content.

Every cell below is real, standalone C, compiled with `gcc` and actually
run to produce the output shown — not asserted, not hand-computed.

## How to install a C kernel for Jupyter

This project already has one (`VAPMIP/notebooks/c/install_c_kernel.sh`),
built on the [`jupyter-c-kernel`](https://pypi.org/project/jupyter-c-kernel/)
package — each notebook cell is compiled as a **standalone** C program
(its own headers, its own `main()`; no shared state between cells) and
executed with `gcc`:

```bash
pip install --quiet --upgrade jupyter notebook
pip install --quiet jupyter-c-kernel
python -m jupyter_c_kernel.install
jupyter kernelspec list        # confirm "c" is now registered
```

Then open this notebook and confirm the kernel selector reads **C**, not
Python 3. Requirements: Python 3.7+, `pip`, `gcc`.


## 1. The β-deepening law

`PtolC/monad.c::monad_learn_ex`, the core per-token update (verbatim
arithmetic, lines ~490–494 of the live file):

```c
double amp = fabs(sin(m->zeros[idx])) * (M_PI * 0.5);
double nb  = m->beta[idx] + E * E * MONAD_ALPHA_LEARN * amp;
if (nb > MONAD_BETA_SAT) nb = MONAD_BETA_SAT;
m->beta[idx] = nb;
```

`MONAD_ALPHA_LEARN` (`0.01`) and `MONAD_BETA_SAT` (`7.552`) are the real
constants, from `PtolC/ptolemy.h` — checked directly, not assumed.
**Flag:** `monad_bin/SPEC.md` states β's range as `(0, 1]`; the live C
constant is `7.552`, not `1.0`. Worth resolving before this section
ships — either the spec is stale or this constant is from a different
generation of the Monad than the one `SPEC.md` documents.

β never resets on its own — it only grows (saturating), which is the
concrete mechanism behind "knowledge deepens with exposure": below is
the same word seen 5 times in a row, extracted standalone (not linked
against the full `Monad` struct — the isolated formula only, same
constants):


In [1]:
/* Standalone extraction of the core beta-update arithmetic from
 * PtolC/monad.c::monad_learn_ex (lines ~490-494), using the real
 * constants from PtolC/ptolemy.h. NOT linked against the full Monad
 * struct -- isolates just the formula for demonstration. */
#include <stdio.h>
#include <math.h>

#define MONAD_ALPHA_LEARN 0.01
#define MONAD_BETA_SAT    7.552

double learn_step(double beta, double E, double zero_gamma) {
    double amp = fabs(sin(zero_gamma)) * (M_PI * 0.5);
    double nb  = beta + E * E * MONAD_ALPHA_LEARN * amp;
    if (nb > MONAD_BETA_SAT) nb = MONAD_BETA_SAT;
    return nb;
}

int main(void) {
    /* A word seen 5 times in a row -- same E, same zero (address fixed) */
    double beta = 0.0;
    double E = 0.42;        /* illustrative spectral energy for one word */
    double gamma = 14.134725; /* first nontrivial Riemann zero, illustrative */

    printf("sight  beta_before   beta_after   delta\n");
    for (int i = 1; i <= 5; i++) {
        double before = beta;
        beta = learn_step(beta, E, gamma);
        printf("%3d    %.6f     %.6f     +%.6f\n", i, before, beta, beta - before);
    }
    return 0;
}


sight  beta_before   beta_after   delta
  1    0.000000     0.002771     +0.002771
  2    0.002771     0.005542     +0.002771
  3    0.005542     0.008313     +0.002771
  4    0.008313     0.011084     +0.002771
  5    0.011084     0.013854     +0.002771


## 2. The `prose_seen` ladder

Also `monad_learn_ex`, lines ~505–512: a word's status ratchets from
`0` (unseen) toward `3` ("verified common" — attested in **both**
WordNet's dictionary sense and real running prose), driven by which kind
of source last taught it:

```c
if (ft == NS_FT_WORDNET)
    m->vocab[idx].prose_seen = 2;              /* canonical dictionary */
else if (ft == NS_FT_PROSE || ...) {
    if (m->vocab[idx].prose_seen == 0)      prose_seen = 1;  /* prose only */
    else if (m->vocab[idx].prose_seen == 2) prose_seen = 3;  /* WN + prose */
}
```

Run standalone, both sighting orders, plus prose-only repeated:


In [1]:
/* Standalone extraction of the prose_seen ladder from
 * PtolC/monad.c::monad_learn_ex (lines ~505-512). ft is the file-type
 * tag the caller passes: NS_FT_WORDNET (the canonical dictionary source)
 * vs NS_FT_PROSE (real running text). */
#include <stdio.h>

typedef enum { FT_PROSE, FT_WORDNET } FType;

int step(int prose_seen, FType ft) {
    if (ft == FT_WORDNET) {
        prose_seen = 2;                         /* canonical dictionary */
    } else {                                    /* FT_PROSE */
        if (prose_seen == 0)      prose_seen = 1;  /* prose only */
        else if (prose_seen == 2) prose_seen = 3;  /* WN + prose = verified */
    }
    return prose_seen;
}

int main(void) {
    const char *NAME[] = {"unseen", "prose-only", "WordNet-only", "verified common"};

    int a = 0;
    printf("word A: WordNet first, then prose\n");
    a = step(a, FT_WORDNET); printf("  after WordNet sighting: %d (%s)\n", a, NAME[a]);
    a = step(a, FT_PROSE);   printf("  after prose sighting:   %d (%s)\n", a, NAME[a]);

    int b = 0;
    printf("\nword B: prose first, then WordNet\n");
    b = step(b, FT_PROSE);   printf("  after prose sighting:   %d (%s)\n", b, NAME[b]);
    b = step(b, FT_WORDNET); printf("  after WordNet sighting: %d (%s)\n", b, NAME[b]);

    int c = 0;
    printf("\nword C: prose only, three times\n");
    for (int i = 0; i < 3; i++) {
        c = step(c, FT_PROSE);
        printf("  sighting %d: %d (%s)\n", i + 1, c, NAME[c]);
    }
    return 0;
}


word A: WordNet first, then prose
  after WordNet sighting: 2 (WordNet-only)
  after prose sighting:   3 (verified common)

word B: prose first, then WordNet
  after prose sighting:   1 (prose-only)
  after WordNet sighting: 2 (WordNet-only)

word C: prose only, three times
  sighting 1: 1 (prose-only)
  sighting 2: 1 (prose-only)
  sighting 3: 1 (prose-only)


**Anomaly, found by just running the real logic, not hunted for:** the
ladder is **order-dependent**, and not symmetrically — WordNet-then-prose
reaches `3` ("verified common"), but prose-then-WordNet only reaches `2`
("WordNet-only"), silently erasing the earlier prose sighting. The
`NS_FT_WORDNET` branch unconditionally overwrites `prose_seen` to `2`
regardless of what it already was, rather than checking "was this
already `1`." Reported here as found, not fixed — genuinely open, worth
a look before this section ships.


## 3. A-matrix coupling — the 2D inverse-distance law

Lines ~538–572: co-occurring tokens gain a directed edge weighted by
**both** spectral distance (how far apart their Riemann-zero addresses
are) and text distance (how many words apart, sliding window):

```c
double d_zero = fabs(m->zeros[sidx[i]] - m->zeros[sidx[j]]) + MONAD_GAP;
double d_text = (double)(j - i);
double w = sE[i] * sE[j] * couple / (d_zero * d_text);
```

`MONAD_GAP` (`0.000707`) again from `ptolemy.h`. **Simplified here,
honestly labeled:** the real function also applies a ×2 bonus when both
tokens share the same "Dirac pole" (`dirac_pole(zeros[i]) ==
dirac_pole(zeros[j])`) — not reproduced below; this shows the base
inverse-distance law only. Three illustrative words:


In [1]:
/* Standalone extraction of the A-matrix coupling law from
 * PtolC/monad.c::monad_learn_ex (lines ~538-572), real MONAD_GAP from
 * ptolemy.h. Simplified: the real function also multiplies by a x2
 * "same Dirac pole" bonus (dirac_pole(zeros[i])==dirac_pole(zeros[j]));
 * that bonus is NOT reproduced here -- this shows the base 2D
 * inverse-distance law only, honestly labeled as a partial extraction. */
#include <stdio.h>
#include <math.h>

#define MONAD_GAP 0.000707

double edge_weight(double E_i, double E_j, double gamma_i, double gamma_j, int d_text) {
    double d_zero = fabs(gamma_i - gamma_j) + MONAD_GAP;
    return E_i * E_j / (d_zero * (double)d_text);
}

int main(void) {
    /* three words in a sentence, illustrative E and gamma values */
    double E[3]     = {0.42, 0.38, 0.51};
    double gamma[3] = {14.134725, 21.022040, 25.010858};
    const char *w[3] = {"crankshaft", "precession", "rotor"};

    printf("pair                    text_dist  zero_dist      weight\n");
    for (int i = 0; i < 3; i++) {
        for (int j = i + 1; j < 3; j++) {
            int d_text = j - i;
            double d_zero = fabs(gamma[i] - gamma[j]) + MONAD_GAP;
            double wgt = edge_weight(E[i], E[j], gamma[i], gamma[j], d_text);
            printf("%-11s <-> %-11s  %6d     %10.6f   %.8f\n",
                   w[i], w[j], d_text, d_zero, wgt);
        }
    }
    return 0;
}


pair                    text_dist  zero_dist      weight
crankshaft  <-> precession        1       6.888022   0.02317066
crankshaft  <-> rotor             2      10.876840   0.00984661
precession  <-> rotor             1       3.989525   0.04857721


## 4. What triggers this, live — not the daemon itself

`monad_bin/SPEC.md` §8 / `PtolC/daemon.c`: the Claude-Code hook
(`monad_observe.py`) ships a prose-sanitised turn to the daemon over a
FIFO, which calls `monad_learn(m, text, w)` (`daemon.c:371/519/581`)
with a fixed weight by message class — `external=1.5` (a user prompt),
`internal=0.9` (assistant prose). That's the entire interaction surface
this notebook needs from the daemon: **what gets sent, and at what
weight** — not how the socket, FIFO, or spool file work, which is
ordinary service plumbing common to most server and desktop software.

## Summary

| piece | status |
|---|---|
| β-deepening law | real C, standalone-extracted, run above — `BETA_SAT` discrepancy flagged against `SPEC.md` |
| `prose_seen` ladder | real C, run above — order-dependence found, not yet fixed |
| A-matrix coupling | real C, run above — simplified (Dirac-pole bonus omitted, labeled) |
| daemon trigger | cited (weights + call site), daemon internals out of scope by design |
| Python twin (`Crank.learn`) | exists, different formula — shown elsewhere in this paper as the CS-paper-readable version, not duplicated here |
